<a href="https://colab.research.google.com/github/oit-cpai/cpai-foundations/blob/main/math/08_linear_algebra_line_fit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M8 線形代数:直線運動の推定から固有値・SVDへ(notebook)

本文:[`08-linear-algebra-line-fit.md`](https://github.com/oit-cpai/cpai-foundations/blob/main/math/08-linear-algebra-line-fit.md)

**進め方**:予想 → 手計算 → コードで確かめる → 実装(`TODO`)→ 条件を変える → 説明。`check(...)` が `OK` と出るまで直してから次に進む。上のセルから順に実行すること。

| 節 | 内容 |
|---|---|
| 1〜4 | 本文の課題1〜4(最小二乗法、ランク、固有値、特異値) |
| 5〜9 | 発展(観測時刻の選び方、時刻の中心化、等加速度運動、リッジ回帰、低ランク近似)。1〜4 が終わってから、興味のあるものを選んでよい |

単位は、時刻を秒 [s]、位置をメートル [m] とする。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def check(name, got, expected, tol=1e-8):
    got, expected = np.asarray(got, dtype=float), np.asarray(expected, dtype=float)
    if got.shape != expected.shape:
        print(f"NG  {name}: 形が違う  got {got.shape}, expected {expected.shape}")
    elif np.allclose(got, expected, atol=tol):
        print(f"OK  {name}")
    else:
        print(f"NG  {name}: got {got}, expected {expected}")

## 1. 課題1:観測から初期位置と速度を求める

一定速度で動く台車の位置を4回測った。モデルは $y_i = p_0 + v t_i + \varepsilon_i$。

| 時刻 $t_i$ [s] | 0 | 1 | 2 | 3 |
|---|---|---|---|---|
| 位置 $y_i$ [m] | 1 | 2 | 4 | 5 |

**予想**:4点をすべて通る直線はあるか。初期位置と速度はおよそいくつか。

- 予想:(ここに書く)

**手計算**:正規方程式 $A^\top A\,\hat{\boldsymbol\theta} = A^\top \boldsymbol y$ を手で解く( $A^\top A$ と $A^\top \boldsymbol y$ を求め、 $2 \times 2$ の方程式を解く)。

- 手計算:(ここに書く)

**実装**:(1)時刻の配列 `t` から行列 $A$( $n \times 2$。1列目がすべて1、2列目が時刻)を作る関数 `design_matrix`、(2)正規方程式を `np.linalg.solve` で解く関数 `fit_normal` を書く。

In [ ]:
t1 = np.array([0.0, 1.0, 2.0, 3.0])
y1 = np.array([1.0, 2.0, 4.0, 5.0])

def design_matrix(t):
    raise NotImplementedError("TODO: 1列目がすべて1、2列目が t の (n, 2) 行列を返す(np.column_stack を使う)")

def fit_normal(A, y):
    raise NotImplementedError("TODO: 正規方程式 A^T A theta = A^T y を np.linalg.solve で解く")

A1 = design_matrix(t1)
theta1 = fit_normal(A1, y1)
print("A =\n", A1)
print("推定値 (p0, v) =", theta1)
check("A の形", A1.shape, (4, 2))
check("A^T A", A1.T @ A1, [[4, 6], [6, 14]])
check("手計算の答え (0.9, 1.4)", theta1, [0.9, 1.4])
check("np.linalg.lstsq と一致", theta1, np.linalg.lstsq(A1, y1, rcond=None)[0])

**残差を確かめる**:残差 $\boldsymbol r = \boldsymbol y - A\hat{\boldsymbol\theta}$ は、 $A$ のすべての列と直交する( $A^\top \boldsymbol r = \boldsymbol 0$ )。図の赤い点線が残差で、**縦方向**のずれである(直線への垂線ではない)。

In [ ]:
r1 = y1 - A1 @ theta1
print("残差 r =", np.round(r1, 6), "  2乗和 =", round(float(r1 @ r1), 6))
check("残差の2乗和 0.2", r1 @ r1, 0.2)
check("A^T r = 0(残差の和も、時刻を掛けた和も 0)", A1.T @ r1, [0.0, 0.0])

tt = np.linspace(-0.3, 3.3, 50)
plt.figure(figsize=(4.5, 3.2))
plt.plot(t1, y1, "o", label="measured")
plt.plot(tt, theta1[0] + theta1[1] * tt, label="least squares")
for ti, yi, ri in zip(t1, y1, r1):
    plt.plot([ti, ti], [yi, yi - ri], "r:")
plt.xlabel("time t [s]"); plt.ylabel("position y [m]"); plt.legend(); plt.grid(); plt.show()

**最小になっていることを確かめる**:推定値を少しずらすと、残差の2乗和は必ず増えるはずである。

In [ ]:
def sse(A, y, theta):
    r = y - A @ theta
    return float(r @ r)

rng = np.random.default_rng(0)
print("ずらすと必ず増えるか:",
      all(sse(A1, y1, theta1 + 0.05 * rng.normal(size=2)) > sse(A1, y1, theta1) for _ in range(200)))

## 2. 課題2:同じ時刻に4回測ったら

4回とも時刻 $t = 1$ s に測ったとする(測った値は $1, 2, 4, 5$ のまま使う)。

**予想**:行列のランクはいくつか。初期位置と速度を別々に決められるか。

- 予想:(ここに書く)

In [ ]:
t_same = np.array([1.0, 1.0, 1.0, 1.0])
A_same = design_matrix(t_same)
print("rank A_same =", np.linalg.matrix_rank(A_same))

# 時刻 1 s の位置が同じ (= 3 m) になる2つの組み合わせ
for theta in [np.array([3.0, 0.0]), np.array([1.0, 2.0]), np.array([-1.0, 4.0])]:
    print(f"(p0, v) = {theta}  →  残差の2乗和 = {sse(A_same, y1, theta):.3f}")

try:
    fit_normal(A_same, y1)
except np.linalg.LinAlgError as e:
    print("正規方程式を solve で解くと → エラー:", e)

残差の2乗和が同じになる組み合わせが無数にある。`np.linalg.lstsq` や擬似逆行列 `np.linalg.pinv` は、その中で**パラメータの長さ(ノルム)が最小のもの**を返す。

**実装**:擬似逆行列で解を求め、(1) $p_0 + v \cdot 1 = 3$ を満たすこと、(2)ノルムが最小であること(手計算で求めた値と一致すること)を確かめる。

In [ ]:
theta_pinv = None  # TODO: np.linalg.pinv(A_same) @ y1
print("擬似逆行列の解 (p0, v) =", theta_pinv)
check("時刻 1 s の位置は 3 m", theta_pinv[0] + theta_pinv[1], 3.0)
check("ノルム最小の解 (1.5, 1.5)", theta_pinv, [1.5, 1.5])

# 同じ時刻の測定を増やしても、ランクは変わらない
for n in [4, 10, 100]:
    print(f"同じ時刻で {n:3d} 回測る → rank = {np.linalg.matrix_rank(design_matrix(np.ones(n)))}")

## 3. 課題3:誤差の等高線と固有値

パラメータの平面(横軸 $p_0$、縦軸 $v$)の上で、残差の2乗和 $J(p_0, v)$ の等高線を描く。

**予想**:課題1と課題2で、等高線の形はどう違うか。

- 予想:(ここに書く)

In [ ]:
def sse_grid(A, y, p0_range, v_range, n=201):
    P0, V = np.meshgrid(np.linspace(*p0_range, n), np.linspace(*v_range, n))
    J = np.zeros_like(P0)
    for i in range(len(y)):
        J += (y[i] - A[i, 0] * P0 - A[i, 1] * V) ** 2
    return P0, V, J

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, (A, title) in zip(axes, [(A1, "task 1: t = 0,1,2,3"), (A_same, "task 2: t = 1,1,1,1")]):
    P0, V, J = sse_grid(A, y1, (-2, 4), (-1, 4))
    cs = ax.contour(P0, V, J, levels=J.min() + np.array([0.1, 1, 2, 5, 10, 20, 40]))  # 最小値からの増え方で描く
    ax.clabel(cs, fontsize=7)
    ax.set_xlabel("p0 [m]"); ax.set_ylabel("v [m/s]"); ax.set_title(title); ax.set_aspect("equal"); ax.grid()
plt.tight_layout(); plt.show()

**実装**: $A^\top A$ の固有値と固有ベクトルを `np.linalg.eigh` で求める関数 `eig_sym` を書く( `eigh` は対称行列用で、固有値を小さい順に返す)。

**手計算**: $A^\top A = \begin{pmatrix} 4 & 6 \\ 6 & 14 \end{pmatrix}$ の固有値を、 $\det(A^\top A - \lambda I) = 0$ から求める。

- 手計算:(ここに書く)

In [ ]:
def eig_sym(M):
    raise NotImplementedError("TODO: np.linalg.eigh で (固有値, 固有ベクトルを列に並べた行列) を返す")

lam1, U1 = eig_sym(A1.T @ A1)
lam_same, U_same = eig_sym(A_same.T @ A_same)
print("課題1:A^T A の固有値 =", np.round(lam1, 4))
print("課題2:A^T A の固有値 =", np.round(lam_same, 4))
check("課題1の固有値 9 ± sqrt(61)", lam1, [9 - np.sqrt(61), 9 + np.sqrt(61)])
check("課題2の固有値 0 と 8", lam_same, [0.0, 8.0])
check("固有ベクトルの定義 M u = lambda u", (A1.T @ A1) @ U1, U1 * lam1)

**固有ベクトルの方向に動かす**:最小点から、各固有ベクトルの方向に距離 $s$ だけ動かすと、誤差は $J(\hat{\boldsymbol\theta}) + \lambda s^2$ だけ増える(固有値 $\lambda$ が大きい方向ほど急に増える)。固有値 0 の方向では、誤差が変わらない。

In [ ]:
s = 0.5
for lam, u in zip(lam1, U1.T):
    J_move = sse(A1, y1, theta1 + s * u)
    print(f"課題1: λ = {lam:7.3f}, 方向 {np.round(u, 3)} に {s} 動かす → J = {J_move:.4f}"
          f"(予想 {sse(A1, y1, theta1) + lam * s**2:.4f})")

u0 = U_same[:, 0]   # 固有値 0 の固有ベクトル
print("課題2: 固有値 0 の方向 =", np.round(u0, 3))
for s in [0.0, 1.0, 3.0]:
    print(f"  pinv の解から {s} 動かす → J = {sse(A_same, y1, theta_pinv + s * u0):.4f}")

# 等高線に固有ベクトルを描く
P0, V, J = sse_grid(A1, y1, (-1, 3), (0, 3))
plt.figure(figsize=(4.5, 4))
plt.contour(P0, V, J, levels=[0.3, 1, 2, 5, 10])
for lam, u in zip(lam1, U1.T):
    plt.arrow(*theta1, *(0.8 * u), head_width=0.06, color="r")
    plt.text(*(theta1 + 0.9 * u), f"λ={lam:.2f}", color="r")
plt.plot(*theta1, "ko"); plt.xlabel("p0 [m]"); plt.ylabel("v [m/s]"); plt.gca().set_aspect("equal"); plt.grid(); plt.show()

楕円の長い軸は、**小さい固有値**の固有ベクトルの方向に伸びる。その方向は、パラメータを変えても誤差が増えにくい(=データから決めにくい)方向である。

## 4. 課題4:短い時間だけ観測すると、速度の推定が不安定になる

本当の値を $p_0 = 1$ m、 $v = 1.4$ m/s、測定誤差を標準偏差 $\sigma = 0.2$ m の正規分布とする。ふつうの時間幅(0, 1, 2, 3 s)と短い時間幅(0, 0.01, 0.02, 0.03 s)で、データを作って推定することを 1000 回くり返す。

**予想**:短い時間幅で、速度の推定のばらつき(標準偏差)は、ふつうの時間幅の何倍になるか。

- 予想:(ここに書く)

**実装**:時刻 `t` で、誤差を変えながらデータを作り、毎回 `np.linalg.lstsq` で推定して、推定値を `(n_trials, 2)` の配列で返す関数 `simulate_estimates` を書く。

In [ ]:
p0_true, v_true, sigma = 1.0, 1.4, 0.2

def simulate_estimates(t, n_trials=1000, seed=0):
    rng = np.random.default_rng(seed)
    A = design_matrix(t)
    est = np.zeros((n_trials, 2))
    for k in range(n_trials):
        raise NotImplementedError("TODO: y = p0_true + v_true t + 誤差(rng.normal(0, sigma, len(t)))を作り、lstsq で推定して est[k] に入れる")
    return est

t_normal = np.array([0.0, 1.0, 2.0, 3.0])
t_short = np.array([0.0, 0.01, 0.02, 0.03])
est_normal = simulate_estimates(t_normal)
est_short = simulate_estimates(t_short)
for name, t, est in [("ふつう", t_normal, est_normal), ("短い", t_short, est_short)]:
    theory = sigma / np.sqrt(np.sum((t - t.mean()) ** 2))
    print(f"{name:3s}: 平均 {np.round(est.mean(axis=0), 3)}, 標準偏差 {np.round(est.std(axis=0), 3)}, "
          f"速度の標準偏差の理論値 {theory:.3f}")
check("ふつうの時間幅:速度の標準偏差は理論値に近い", est_normal[:, 1].std(), sigma / np.sqrt(5), tol=0.01)
check("短い時間幅:速度の標準偏差は理論値に近い", est_short[:, 1].std(), sigma / np.sqrt(5e-4), tol=0.6)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
for ax, est, title in zip(axes, [est_normal, est_short], ["t = 0, 1, 2, 3 s", "t = 0, 0.01, 0.02, 0.03 s"]):
    ax.plot(est[:, 0], est[:, 1], ".", ms=2, alpha=0.5)
    ax.plot(p0_true, v_true, "r+", ms=12, mew=2)
    ax.set_xlabel("estimated p0 [m]"); ax.set_ylabel("estimated v [m/s]"); ax.set_title(title); ax.grid()
plt.tight_layout(); plt.show()

**特異値を比べる**:同じ時刻・短い時間幅・ふつうの時間幅の3つで、 $A$ の特異値を求める。 $A^\top A$ の固有値が特異値の2乗になることも確かめる。

In [ ]:
for name, t in [("同じ時刻", t_same), ("短い時間幅", t_short), ("ふつうの時間幅", t_normal)]:
    A = design_matrix(t)
    sv = np.linalg.svd(A, compute_uv=False)
    eig = np.sort(np.linalg.eigvalsh(A.T @ A))[::-1]
    print(f"{name:8s}: 特異値 {np.round(sv, 4)}   A^T A の固有値 {np.round(eig, 4)}   特異値の2乗 {np.round(sv**2, 4)}")

**擬似逆行列は小さな特異値で誤差を増幅する**: $A = U\Sigma V^\top$ のとき、 $\hat{\boldsymbol\theta} = \sum_k \frac{\boldsymbol u_k^\top \boldsymbol y}{s_k} \boldsymbol v_k$ である。

**実装**:SVD を使って最小二乗解を求める関数 `fit_svd` を書く(`np.linalg.svd(A, full_matrices=False)` を使い、特異値が `tol` 以下の項は捨てる)。

In [ ]:
def fit_svd(A, y, tol=1e-10):
    U, s, Vt = np.linalg.svd(A, full_matrices=False)
    raise NotImplementedError("TODO: s > tol の k について (U[:, k] @ y) / s[k] * Vt[k] を足し合わせる")

check("課題1:fit_svd = lstsq", fit_svd(A1, y1), theta1)
check("課題2:fit_svd = pinv(ノルム最小の解)", fit_svd(A_same, y1), [1.5, 1.5])

# 短い時間幅:誤差のベクトルが、どの特異値の方向にどれだけ増幅されるか
U, s, Vt = np.linalg.svd(design_matrix(t_short), full_matrices=False)
print("短い時間幅の特異値:", np.round(s, 4), " → 逆数(増幅率):", np.round(1 / s, 1))

## 5. 発展1:4回の観測を、いつ行うのがよいか

観測は4回、時刻は 0〜3 s の範囲で選べるとする。速度をいちばん正確に推定するには、いつ測ればよいか。

**予想**:等間隔(0, 1, 2, 3)と、両端に2回ずつ(0, 0, 3, 3)と、真ん中に集める(1, 1.5, 1.5, 2)では、どれがよいか。

- 予想:(ここに書く)

速度の推定の分散は、 $\sigma^2 (A^\top A)^{-1}$ の (2, 2) 成分である(本文 §5 の式 $\sigma^2 / \sum_i (t_i - \bar t)^2$ と同じ)。

**実装**:時刻の配列から、速度の推定の標準偏差の理論値を `(A^T A)^{-1}` を使って求める関数 `v_std_theory` を書く(ここは理論式の確認なので、逆行列を使ってよい)。

In [ ]:
def v_std_theory(t, sigma=0.2):
    raise NotImplementedError("TODO: A = design_matrix(t)、C = sigma**2 * inv(A^T A) とし、sqrt(C[1, 1]) を返す")

designs = {"等間隔 0,1,2,3": [0, 1, 2, 3], "両端 0,0,3,3": [0, 0, 3, 3], "真ん中 1,1.5,1.5,2": [1, 1.5, 1.5, 2]}
for name, t in designs.items():
    t = np.array(t, dtype=float)
    sv = np.linalg.svd(design_matrix(t), compute_uv=False)
    print(f"{name:18s}: 理論 {v_std_theory(t):.4f}  シミュレーション {simulate_estimates(t)[:, 1].std():.4f}"
          f"  det(A^T A) = {np.linalg.det(design_matrix(t).T @ design_matrix(t)):6.2f}  最小の特異値 {sv[-1]:.3f}")
check("等間隔の理論値 0.2/sqrt(5)", v_std_theory(np.array([0, 1, 2, 3.0])), 0.2 / np.sqrt(5))
check("両端の理論値 0.2/3", v_std_theory(np.array([0, 0, 3, 3.0])), 0.2 / 3)

**考える**:両端に集めると速度の推定は正確になる。では、等間隔に測ることには、どんな良さがあるか(ヒント:本当の運動が一定速度でなかったら、気づけるのはどちらか。発展3)。

- 考えたこと:(ここに書く)

このように、観測や実験の条件を「推定の精度がよくなるように」選ぶことを**実験計画**という。 $\det(A^\top A)$ を大きくする選び方は D 最適計画と呼ばれる。

## 6. 発展2:時刻の原点をずらす(中心化)と、何が変わり、何が変わらないか

時刻を $t' = t - \bar t$( $\bar t$ は観測時刻の平均)に置き換えると、モデルは $y = c + v t'$ になる。 $c$ は「観測の真ん中の時刻での位置」である。

**予想**:短い時間幅で中心化すると、(a) $A^\top A$ の固有値の比(条件数)、(b)速度の推定の標準偏差、はそれぞれどうなるか。

- 予想:(ここに書く)

In [ ]:
for name, t in [("ふつうの時間幅", t_normal), ("短い時間幅", t_short)]:
    for label, tc in [("そのまま", t), ("中心化", t - t.mean())]:
        A = design_matrix(tc)
        M = A.T @ A
        lam = np.linalg.eigvalsh(M)
        print(f"{name:8s} {label:5s}: A^T A = {np.round(M, 4).tolist()},  条件数 = {lam[-1] / lam[0]:10.1f},"
              f"  速度の標準偏差 = {v_std_theory(tc):.3f}")

中心化すると $A^\top A$ が対角行列になり、2つのパラメータの推定が互いに影響しなくなる。ふつうの時間幅では、条件数(数値計算のしにくさ)が約 14 から約 1.2 に良くなる。短い時間幅では、中心化しても条件数は約 8000 のままである(時刻の値そのものが小さいため)。

どちらの場合も、**速度の推定の標準偏差は変わらない**。短い時間幅で速度がわかりにくいのは、計算の方法の問題ではなく、データに含まれる情報が少ないからである。

**確かめる**:時刻の単位を秒からミリ秒に変えると(`t_short * 1000`)、特異値や条件数はどう変わるか。速度の推定の標準偏差は(単位を m/s にそろえて比べると)変わるか。

In [ ]:
# TODO: t_short をミリ秒にした時刻で、特異値・条件数・速度の標準偏差(単位は m/ms → m/s に直して比べる)を調べる

単位を変えると特異値や条件数は変わる(約 8000 → 約 31)が、速度の推定の精度は変わらない。**条件数は、表し方(単位・原点)によって変わる「計算のしやすさ」**、**推定のばらつきは、データの情報の量で決まる「統計的な精度」**であり、2つを区別する。特異値や等高線を条件どうしで比べるときは、単位と表し方をそろえる(本文 §5.3 の注意)。

## 7. 発展3:一定速度でなかったら(等加速度運動)

本当は加速度 $a$ があり、 $p(t) = p_0 + v t + \frac{1}{2} a t^2$ で動いているとする。

**実装**:(1)3列目に $t^2/2$ を加えた $n \times 3$ の行列を作る関数 `design_matrix_acc`、(2)時刻 0, 0.5, ..., 3 s(7点)でデータを作り、一定速度のモデルと等加速度のモデルの両方で推定して、残差を比べる。

In [ ]:
def design_matrix_acc(t):
    raise NotImplementedError("TODO: 列が 1, t, t^2/2 の (n, 3) 行列を返す")

rng = np.random.default_rng(1)
t7 = np.linspace(0, 3, 7)
a_true = 0.8
y7 = p0_true + v_true * t7 + 0.5 * a_true * t7**2 + rng.normal(0, sigma, len(t7))

theta_cv = np.linalg.lstsq(design_matrix(t7), y7, rcond=None)[0]
theta_ca = np.linalg.lstsq(design_matrix_acc(t7), y7, rcond=None)[0]
print("一定速度のモデル (p0, v)     =", np.round(theta_cv, 3))
print("等加速度のモデル (p0, v, a)  =", np.round(theta_ca, 3), "  本当は", (p0_true, v_true, a_true))
check("行列の形", design_matrix_acc(t7).shape, (7, 3))

In [ ]:
r_cv = y7 - design_matrix(t7) @ theta_cv
r_ca = y7 - design_matrix_acc(t7) @ theta_ca
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
tt = np.linspace(0, 3, 100)
axes[0].plot(t7, y7, "o", label="measured")
axes[0].plot(tt, design_matrix(tt) @ theta_cv, label="constant velocity")
axes[0].plot(tt, design_matrix_acc(tt) @ theta_ca, label="constant acceleration")
axes[0].set_xlabel("t [s]"); axes[0].set_ylabel("y [m]"); axes[0].legend(); axes[0].grid()
axes[1].plot(t7, r_cv, "o-", label="residual (const. velocity)")
axes[1].plot(t7, r_ca, "s-", label="residual (const. acceleration)")
axes[1].axhline(0, color="k", lw=0.5); axes[1].set_xlabel("t [s]"); axes[1].set_ylabel("residual [m]"); axes[1].legend(); axes[1].grid()
plt.tight_layout(); plt.show()
print("残差の2乗和:一定速度", round(float(r_cv @ r_cv), 3), "  等加速度", round(float(r_ca @ r_ca), 3))

# 2種類の時刻しかないと、3つの未知数は決まらない
for t in [np.array([0, 0, 3, 3.0]), np.array([0, 1, 2, 3.0])]:
    print(f"時刻 {t} → 等加速度の行列のランク = {np.linalg.matrix_rank(design_matrix_acc(t))}")

**考える**:一定速度のモデルの残差には、どんな形の偏りがあるか(上に凸か、下に凸か)。残差の**形**を見ることが、モデルが合っていないことに気づく手がかりになる。また、発展1の「両端に2回ずつ」の測り方では、加速度があるかどうかを確かめられない。その理由を、ランクで説明せよ。

- 考えたこと:(ここに書く)

## 8. 発展4:リッジ回帰(正則化)で、小さな特異値の影響を抑える

短い時間幅では、速度の推定が大きくばらついた。そこで、パラメータが大きくなりすぎないよう罰を加えて

$$
J_\lambda(\boldsymbol\theta) = \lVert \boldsymbol y - A\boldsymbol\theta \rVert^2 + \lambda \lVert \boldsymbol\theta \rVert^2
$$

を最小にする(**リッジ回帰**)。解は $(A^\top A + \lambda I)\,\hat{\boldsymbol\theta} = A^\top \boldsymbol y$ を満たす。SVD で書くと、 $\frac{1}{s_k}$ が $\frac{s_k}{s_k^2 + \lambda}$ に置き換わる。 $s_k$ が小さいと、この値は 0 に近づく。

**実装**:リッジ回帰の解を返す関数 `fit_ridge(A, y, lam)` を書く。

In [ ]:
def fit_ridge(A, y, lam):
    raise NotImplementedError("TODO: (A^T A + lam I) theta = A^T y を np.linalg.solve で解く")

check("lam = 0 なら最小二乗解と同じ", fit_ridge(A1, y1, 0.0), theta1)
U, s, Vt = np.linalg.svd(A1, full_matrices=False)
check("SVD の式と一致", fit_ridge(A1, y1, 0.5), sum((s[k] / (s[k]**2 + 0.5)) * (U[:, k] @ y1) * Vt[k] for k in range(2)))

**予想**:短い時間幅で $\lambda$ を大きくすると、速度の推定の(a)ばらつき、(b)平均と本当の値 1.4 のずれ(偏り)は、それぞれどうなるか。

- 予想:(ここに書く)

In [ ]:
A_s = design_matrix(t_short)
rng = np.random.default_rng(0)
Y = p0_true + v_true * t_short + rng.normal(0, sigma, (1000, len(t_short)))
print(" lambda   速度の平均   速度の標準偏差   2乗誤差の平均(速度)")
for lam in [0.0, 1e-4, 1e-3, 1e-2, 1e-1]:
    v_hat = np.array([fit_ridge(A_s, y, lam)[1] for y in Y])
    print(f"{lam:7.0e}   {v_hat.mean():9.3f}   {v_hat.std():12.3f}   {np.mean((v_hat - v_true)**2):14.3f}")

$\lambda$ を大きくすると、ばらつきは小さくなるが、推定値は 0 の方へ引き寄せられ、偏りが生まれる(**偏りとばらつきのトレードオフ**)。速度の情報がもともとデータに少ないので、正則化は「情報を増やす」のではなく「わからない方向を 0 に近い値で埋める」ことに注意する。機械学習のリッジ回帰(S4)、ロボットの減衰つき疑似逆行列(K6)は、同じ考え方である。

## 9. 発展5:たくさんの台車の軌跡と、低ランク近似

20台の台車が、それぞれ違う初期位置と速度で一定速度で動き、30の時刻で位置を測ったとする。位置を並べた $20 \times 30$ の行列 $Y$( $Y_{kj}$ は台車 $k$ の時刻 $t_j$ の位置)を作る。

**予想**:誤差がなければ、 $Y$ のランクはいくつか。誤差があると、特異値はどう並ぶか。

- 予想:(ここに書く)

In [ ]:
rng = np.random.default_rng(2)
n_cart, n_time = 20, 30
t30 = np.linspace(0, 3, n_time)
p0s = rng.uniform(0, 2, n_cart)
vs = rng.uniform(0.5, 2, n_cart)
Y_true = p0s[:, None] + vs[:, None] * t30[None, :]       # (20, 30)
Y_obs = Y_true + rng.normal(0, sigma, Y_true.shape)
print("誤差なしのランク:", np.linalg.matrix_rank(Y_true), "  誤差ありのランク:", np.linalg.matrix_rank(Y_obs))

sv = np.linalg.svd(Y_obs, compute_uv=False)
plt.figure(figsize=(4.5, 3))
plt.semilogy(np.arange(1, len(sv) + 1), sv, "o-")
plt.xlabel("index k"); plt.ylabel("singular value s_k"); plt.grid(); plt.show()
print("大きい順の特異値(最初の5つ):", np.round(sv[:5], 2))

**実装**:大きい方から $r$ 個の特異値だけを残した近似 $Y_r = \sum_{k=1}^{r} s_k \boldsymbol u_k \boldsymbol v_k^\top$ を返す関数 `low_rank` を書く。 $r$ を 1〜5 と変え、近似 $Y_r$ と**本当の軌跡** `Y_true` との誤差(要素ごとの2乗平均の平方根)を調べる。

In [ ]:
def low_rank(Y, r):
    U, s, Vt = np.linalg.svd(Y, full_matrices=False)
    raise NotImplementedError("TODO: U[:, :r] @ diag(s[:r]) @ Vt[:r] を返す")

def rmse(X, Z):
    return float(np.sqrt(np.mean((X - Z) ** 2)))

print("観測そのものの誤差(RMSE):", round(rmse(Y_obs, Y_true), 4))
for r in [1, 2, 3, 5]:
    print(f"r = {r}: 近似と本当の軌跡の RMSE = {rmse(low_rank(Y_obs, r), Y_true):.4f}")
check("r = 2 の近似のランク", np.linalg.matrix_rank(low_rank(Y_obs, 2)), 2)

誤差のない軌跡は、すべて「定数」と「時刻に比例する量」の組み合わせなので、ランクは2である。 $r = 2$ で近似すると、誤差の大部分が取り除かれ、観測そのものより本当の軌跡に近くなる。 $r$ を増やしすぎると、誤差まで近似してしまう。主成分分析(PCA)や、データの次元削減は、この考え方を使う。

**考える**: $r = 2$ のときの $V$ の最初の2行(`Vt[:2]`)を時刻に対して描くと、どんな形か。それは、この題材の何に対応しているか。

- 考えたこと:(ここに書く)

## 10. 条件を変える

1. 課題1で、測定値を1つだけ大きく変える(例: $y_4 = 8$)。推定値と残差はどう変わるか。1つの外れた値の影響が大きいのはなぜか
2. 課題4で、観測の回数を4回から40回(同じ時間幅の中で等間隔)に増やす。速度の標準偏差は何倍になるか。理論式と比べよ
3. 発展4で、ふつうの時間幅に同じ $\lambda$ を使うと、偏りはどのくらいか。短い時間幅と比べて、正則化の影響が小さいのはなぜか(特異値で説明する)

## 11. 説明(自分の言葉で書く)

1. 4つの観測があるのに、2つの未知数を決められないことがあるのはなぜか
   - (ここに書く)
2. 正定値と半正定値の違いは、この題材では何として現れたか
   - (ここに書く)
3. ただ1つに決まった推定値でも、測定の誤差に弱いことがあるのはなぜか。特異値を使って説明せよ
   - (ここに書く)
4. 時刻を中心化すると条件数は良くなったが、速度の推定の精度は変わらなかった。この2つの違いは何か
   - (ここに書く)